In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from dotenv import load_dotenv

load_dotenv()

connection_url = URL.create(
    'mssql+pyodbc',
    username=os.getenv('SQL_USER'),
    password=os.getenv("SQL_PASSWORD"),
    host=os.getenv("SQL_SERVER"),
    database=os.getenv("SQL_DATABASE"),
    query={"driver": "ODBC Driver 18 for SQL Server"},
)
engine = create_engine(connection_url, fast_executemany=True)

In [ ]:
df = pd.read_csv("../data/processed/ab_test_clean.csv")
df.to_sql("ab_test_users", engine, schema="dbo", if_exists="append",
          index=False, chunksize=10000)

In [ ]:
# check the load
with engine.connect() as conn:
    print(conn.execute(text("SELECT COUNT(*) FROM dbo.ab_test_users")).scalar())
    result = conn.execute(text("""
        SELECT test_group, COUNT(*) AS users, SUM(CAST(converted AS INT)) AS conversions
        FROM dbo.ab_test_users
        GROUP BY test_group
    """))
    for row in result:
        print(row)